# Polynomial Regression - House Price Prediction

This notebook demonstrates polynomial regression for predicting house prices based on square footage.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully")

## 2. Load and Explore Data

In [ ]:
# Load dataset
data_path = Path("house_prices.csv")
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at: {data_path.resolve()}")

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Data Info
print("Data Types and Missing Values:")
print(df.info())

print("\nStatistical Summary:")
df.describe()

## 3. Data Visualization

In [ ]:
# Create visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Square Footage Distribution
axes[0].hist(df['sqft'], bins=15, color='skyblue', edgecolor='black')
axes[0].set_xlabel('Square Footage')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of House Square Footage')
axes[0].grid(alpha=0.3)

# Price Distribution
axes[1].hist(df['price'], bins=15, color='lightcoral', edgecolor='black')
axes[1].set_xlabel('Price ($K)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of House Prices')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Average Square Footage: {df['sqft'].mean():.0f}")
print(f"Average Price: ${df['price'].mean():.1f}K")

In [ ]:
# Scatter plot: Square Footage vs Price
plt.figure(figsize=(12, 6))
plt.scatter(df['sqft'], df['price'], alpha=0.6, s=80, color='steelblue')
plt.xlabel('Square Footage', fontsize=12)
plt.ylabel('Price ($K)', fontsize=12)
plt.title('House Square Footage vs Price', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate correlation
correlation = df['sqft'].corr(df['price'])
print(f"Correlation between Square Footage and Price: {correlation:.3f}")

## 4. Prepare Data for Modeling

In [ ]:
# Separate features and target
X = df[['sqft']].values
y = df['price'].values

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTrain set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## 5. Polynomial Feature Engineering

In [ ]:
# Create polynomial features (degree 2)
degree = 2
poly = PolynomialFeatures(degree=degree)

# Fit and transform training data
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

print(f"Original features: {X_train.shape[1]}")
print(f"Polynomial features (degree {degree}): {X_train_poly.shape[1]}")

# Get feature names
feature_names = poly.get_feature_names_out(['sqft'])
print(f"\nFeature names: {feature_names}")

# Show transformed features
print(f"\nFirst 3 samples - transformed features:")
print(X_train_poly[:3])

## 6. Feature Scaling

In [ ]:
# Scale features for better numerical stability
scaler = StandardScaler()
X_train_poly_scaled = scaler.fit_transform(X_train_poly)
X_test_poly_scaled = scaler.transform(X_test_poly)

print(f"Scaled training features - Mean: {X_train_poly_scaled.mean():.4f}, Std: {X_train_poly_scaled.std():.4f}")
print(f"Scaled test features - Mean: {X_test_poly_scaled.mean():.4f}, Std: {X_test_poly_scaled.std():.4f}")

## 7. Train Polynomial Regression Model

In [ ]:
# Create and train model
model = LinearRegression()
model.fit(X_train_poly_scaled, y_train)

print("✓ Model trained successfully!")
print(f"\nModel Intercept: {model.intercept_:.4f}")
print(f"Model Coefficients: {model.coef_}")

# Create coefficient dataframe
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': model.coef_
})

print("\nCoefficients:")
print(coef_df)

## 8. Make Predictions

In [ ]:
# Training predictions
y_train_pred = model.predict(X_train_poly_scaled)

# Test predictions
y_test_pred = model.predict(X_test_poly_scaled)

print("Sample Predictions:")
pred_comparison = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_test_pred,
    'Error': y_test - y_test_pred,
    'Error %': (abs(y_test - y_test_pred) / y_test * 100).round(2)
})

print(pred_comparison)

## 9. Model Evaluation

In [ ]:
# Calculate metrics for training set
train_mae = mean_absolute_error(y_train, y_train_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_r2 = r2_score(y_train, y_train_pred)

# Calculate metrics for test set
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_test_pred)

print("TRAINING METRICS:")
print(f"  MAE:  ${train_mae:.2f}K")
print(f"  RMSE: ${train_rmse:.2f}K")
print(f"  R²:   {train_r2:.3f}")

print("\nTEST METRICS:")
print(f"  MAE:  ${test_mae:.2f}K")
print(f"  RMSE: ${test_rmse:.2f}K")
print(f"  R²:   {test_r2:.3f}")

print(f"\nOverfitting Check:")
print(f"  Train R² - Test R² = {(train_r2 - test_r2):.3f}")
if abs(train_r2 - test_r2) < 0.1:
    print("  ✓ Good generalization (low overfitting)")
else:
    print("  ⚠ Possible overfitting")

## 10. Cross-Validation

In [ ]:
# Prepare all data for cross-validation
X_all_poly = poly.transform(X)
X_all_poly_scaled = scaler.transform(X_all_poly)

# 5-fold cross-validation
cv_scores = cross_val_score(LinearRegression(), X_all_poly_scaled, y, cv=5, scoring='r2')

print(f"5-Fold Cross-Validation R² Scores: {cv_scores}")
print(f"Mean CV Score: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

## 11. Visualize Model Fit

In [ ]:
# Create smooth curve for prediction
X_range = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
X_range_poly = poly.transform(X_range)
X_range_poly_scaled = scaler.transform(X_range_poly)
y_range_pred = model.predict(X_range_poly_scaled)

# Plot
plt.figure(figsize=(14, 6))
plt.scatter(X, y, alpha=0.6, s=80, label='Actual Data', color='steelblue')
plt.plot(X_range, y_range_pred, 'r-', linewidth=2.5, label='Polynomial Fit (Degree 2)')
plt.xlabel('Square Footage', fontsize=12)
plt.ylabel('Price ($K)', fontsize=12)
plt.title('Polynomial Regression: House Prices', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Residual Analysis

In [ ]:
# Calculate residuals
residuals_train = y_train - y_train_pred
residuals_test = y_test - y_test_pred

# Create residual plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Residuals vs Predicted (Train)
axes[0, 0].scatter(y_train_pred, residuals_train, alpha=0.6, color='green')
axes[0, 0].axhline(y=0, color='r', linestyle='--')
axes[0, 0].set_xlabel('Predicted Price')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Training: Residuals vs Predicted')
axes[0, 0].grid(True, alpha=0.3)

# Residuals vs Predicted (Test)
axes[0, 1].scatter(y_test_pred, residuals_test, alpha=0.6, color='orange')
axes[0, 1].axhline(y=0, color='r', linestyle='--')
axes[0, 1].set_xlabel('Predicted Price')
axes[0, 1].set_ylabel('Residuals')
axes[0, 1].set_title('Test: Residuals vs Predicted')
axes[0, 1].grid(True, alpha=0.3)

# Residual Distribution (Train)
axes[1, 0].hist(residuals_train, bins=10, color='green', alpha=0.7, edgecolor='black')
axes[1, 0].set_xlabel('Residuals')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Training: Residual Distribution')
axes[1, 0].grid(True, alpha=0.3)

# Residual Distribution (Test)
axes[1, 1].hist(residuals_test, bins=10, color='orange', alpha=0.7, edgecolor='black')
axes[1, 1].set_xlabel('Residuals')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Test: Residual Distribution')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Residuals Mean (Train): {residuals_train.mean():.4f}")
print(f"Residuals Std (Train): {residuals_train.std():.4f}")
print(f"Residuals Mean (Test): {residuals_test.mean():.4f}")
print(f"Residuals Std (Test): {residuals_test.std():.4f}")

## 13. Make Single Prediction

In [ ]:
# Predict price for a house with specific square footage
new_sqft = np.array([[2500]])
new_sqft_poly = poly.transform(new_sqft)
new_sqft_poly_scaled = scaler.transform(new_sqft_poly)
predicted_price = model.predict(new_sqft_poly_scaled)[0]

print(f"House with {new_sqft[0][0]:.0f} sqft:")
print(f"Predicted Price: ${predicted_price*1000:,.2f}")

# Show prediction breakdown
features = poly.get_feature_names_out(['sqft'])
breakdown = pd.DataFrame({
    'Feature': features,
    'Feature Value': new_sqft_poly[0],
    'Coefficient': model.coef_,
    'Contribution': new_sqft_poly[0] * model.coef_
})

print("\nPrediction Breakdown:")
print(breakdown)
print(f"\nIntercept: {model.intercept_:.4f}")
print(f"Total Prediction: ${predicted_price*1000:,.2f}")

## 14. Save Model

In [ ]:
# Save model and preprocessing objects
model_data = {
    'model': model,
    'poly': poly,
    'scaler': scaler,
    'degree': degree,
    'metrics': {
        'test_mae': test_mae,
        'test_rmse': test_rmse,
        'test_r2': test_r2
    }
}

joblib.dump(model_data, 'polynomial_regression_model.pkl')
print("✓ Model saved to 'polynomial_regression_model.pkl'")

## 15. Summary and Insights

In [ ]:
print("=" * 60)
print("POLYNOMIAL REGRESSION - HOUSE PRICE PREDICTION SUMMARY")
print("=" * 60)

print("\n📊 DATASET:")
print(f"  - Total samples: {len(df)}")
print(f"  - Features: Square Footage")
print(f"  - Target: Price ($K)")

print("\n🔧 MODEL CONFIGURATION:")
print(f"  - Polynomial Degree: {degree}")
print(f"  - Features Generated: {len(feature_names)}")
print(f"  - Scaling: StandardScaler")

print("\n📈 TEST PERFORMANCE:")
print(f"  - MAE: ${test_mae:.2f}K")
print(f"  - RMSE: ${test_rmse:.2f}K")
print(f"  - R² Score: {test_r2:.3f}")

print("\n✅ KEY INSIGHTS:")
print(f"  - Price relationship with square footage is {degree}-order polynomial")
print(f"  - Model explains {test_r2*100:.1f}% of price variation")
print(f"  - Average prediction error: ${test_mae*1000:,.0f}")

print("\n" + "=" * 60)